[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gaurav14cs17/Multimodal-Deep-Learning/blob/main/02_Vision_Language_Models/02_image_captioning.ipynb)

# 02. Image Captioning: Image → Text Generation

**This notebook covers:**
- Image captioning architecture (encoder-decoder)
- Building a small captioning model from scratch
- Using pretrained models (BLIP) for captioning
- Visualizing attention: which image regions generate which words

---

In [ ]:
# ============================================================
#  Colab Setup (run this cell first if on Google Colab)
# ============================================================
import os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_URL = "https://github.com/Gaurav14cs17/Multimodal-Deep-Learning.git"
    REPO_DIR = "/content/Multimodal-Deep-Learning"

    if not os.path.exists(REPO_DIR):
        !git clone {REPO_URL} {REPO_DIR}
        !pip install -q -r {REPO_DIR}/requirements.txt

    os.chdir(f"{REPO_DIR}/02_Vision_Language_Models")
    os.makedirs(f"{REPO_DIR}/assets", exist_ok=True)
    print(f"Colab ready — working in {os.getcwd()}")
else:
    os.makedirs("../assets", exist_ok=True)

In [ ]:
import sys
sys.path.append('..')

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
from utils.visualization import *
from utils.helpers import count_parameters

set_style()

## 1. Architecture: Encoder-Decoder for Captioning

```
Image → Vision Encoder → Image Features (K, V)
                                    ↓
[BOS] → Text Decoder (Q) ← Cross-Attention → "a" → "cat" → "sitting" → [EOS]
```

The decoder generates one word at a time, attending to image features.

In [ ]:
# Visualize the captioning architecture
fig, ax = plt.subplots(figsize=(14, 8))
ax.set_xlim(0, 14)
ax.set_ylim(0, 8)
ax.axis('off')
ax.set_title('Image Captioning Architecture', fontsize=18, fontweight='bold', pad=20)

# Image side
draw_architecture_block(ax, 3, 7, 3.5, 0.8, 'Image', '#E74C3C')
draw_architecture_block(ax, 3, 5.5, 3.5, 0.8, 'Vision Encoder (ViT)', '#E74C3C')
draw_architecture_block(ax, 3, 4, 3.5, 0.8, 'Image Features\n[N_patches, D]', '#C0392B')

draw_arrow(ax, (3, 6.5), (3, 6.0))
draw_arrow(ax, (3, 5.0), (3, 4.5))

# Decoder side
draw_architecture_block(ax, 10, 7, 4, 0.8, 'Text Tokens (shifted)', '#3498DB')
draw_architecture_block(ax, 10, 5.5, 4, 0.8, 'Self-Attention (causal)', '#3498DB')
draw_architecture_block(ax, 10, 4, 4, 0.8, 'Cross-Attention\nQ=text, K/V=image', '#9B59B6')
draw_architecture_block(ax, 10, 2.5, 4, 0.8, 'FFN + Softmax', '#2ECC71')
draw_architecture_block(ax, 10, 1, 4, 0.8, 'Next Word Prediction', '#F39C12')

draw_arrow(ax, (10, 6.5), (10, 6.0))
draw_arrow(ax, (10, 5.0), (10, 4.5))
draw_arrow(ax, (10, 3.5), (10, 3.0))
draw_arrow(ax, (10, 2.0), (10, 1.5))

# Cross-attention connection
draw_arrow(ax, (4.8, 4.0), (7.8, 4.0), color='#9B59B6', lw=2.5)
ax.text(6.3, 4.4, 'K, V', fontsize=11, color='#9B59B6', fontweight='bold')

plt.tight_layout()
plt.savefig('../assets/captioning_architecture.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Build a small captioning model from scratch

class CaptioningDecoder(nn.Module):
    """Autoregressive decoder with cross-attention to image features."""
    def __init__(self, vocab_size=1000, embed_dim=128, n_heads=4,
                 n_layers=3, max_len=32):
        super().__init__()
        self.token_embed = nn.Embedding(vocab_size, embed_dim)
        self.pos_embed = nn.Embedding(max_len, embed_dim)
        
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=embed_dim, nhead=n_heads,
            dim_feedforward=embed_dim * 4, batch_first=True
        )
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=n_layers)
        self.output_proj = nn.Linear(embed_dim, vocab_size)
        self.embed_dim = embed_dim

    def forward(self, tgt_ids, memory):
        """Forward pass.
        tgt_ids: [B, T] text token ids
        memory:  [B, N, D] image features from encoder
        """
        B, T = tgt_ids.shape
        pos = torch.arange(T, device=tgt_ids.device).unsqueeze(0).expand(B, -1)
        x = self.token_embed(tgt_ids) + self.pos_embed(pos)
        
        # Causal mask (prevent attending to future tokens)
        causal_mask = nn.Transformer.generate_square_subsequent_mask(T, device=x.device)
        
        # Decode with cross-attention to image features
        output = self.decoder(x, memory, tgt_mask=causal_mask)
        logits = self.output_proj(output)  # [B, T, vocab_size]
        return logits

    @torch.no_grad()
    def generate(self, memory, max_len=20, bos_id=1, eos_id=2):
        """Autoregressive generation."""
        B = memory.shape[0]
        generated = torch.full((B, 1), bos_id, dtype=torch.long, device=memory.device)
        
        for _ in range(max_len):
            logits = self.forward(generated, memory)
            next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)
            generated = torch.cat([generated, next_token], dim=1)
            if (next_token == eos_id).all():
                break
        return generated


# Simple vision encoder (reuse from before)
class SimpleVisionEncoder(nn.Module):
    def __init__(self, img_size=32, patch_size=4, embed_dim=128, n_layers=2):
        super().__init__()
        n_patches = (img_size // patch_size) ** 2
        self.patch_embed = nn.Conv2d(3, embed_dim, patch_size, patch_size)
        self.pos_embed = nn.Parameter(torch.randn(1, n_patches, embed_dim) * 0.02)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=4, dim_feedforward=embed_dim*4, batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x):
        x = self.patch_embed(x).flatten(2).transpose(1, 2)
        x = x + self.pos_embed
        return self.norm(self.encoder(x))  # [B, N_patches, D]


class ImageCaptioningModel(nn.Module):
    def __init__(self, vocab_size=1000, embed_dim=128):
        super().__init__()
        self.encoder = SimpleVisionEncoder(embed_dim=embed_dim)
        self.decoder = CaptioningDecoder(vocab_size=vocab_size, embed_dim=embed_dim)

    def forward(self, images, caption_ids):
        memory = self.encoder(images)
        return self.decoder(caption_ids, memory)

    @torch.no_grad()
    def generate(self, images, **kwargs):
        memory = self.encoder(images)
        return self.decoder.generate(memory, **kwargs)


model = ImageCaptioningModel(vocab_size=500, embed_dim=128)
count_parameters(model)

# Test
imgs = torch.randn(2, 3, 32, 32)
caps = torch.randint(0, 500, (2, 10))
logits = model(imgs, caps)
print(f"\nInput: images {imgs.shape}, captions {caps.shape}")
print(f"Output logits: {logits.shape} (predict next token at each position)")

In [ ]:
# Visualize: Autoregressive generation process

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
fig.suptitle('Autoregressive Caption Generation (step by step)', 
             fontsize=14, fontweight='bold')

steps = [
    ('[BOS]', '→ a'),
    ('[BOS] a', '→ cute'),
    ('[BOS] a cute', '→ cat'),
    ('[BOS] a cute cat', '→ sitting'),
    ('[BOS] a cute cat sitting', '→ [EOS]'),
]

for ax, (context, prediction) in zip(axes, steps):
    ax.axis('off')
    # Show image
    ax.imshow(np.random.rand(8, 8, 3) * 0.3 + 0.4, extent=[0, 1, 0.5, 1.2])
    
    # Show context and prediction
    ax.text(0.5, 0.35, context, ha='center', fontsize=8, 
            bbox=dict(boxstyle='round', facecolor='#3498DB', alpha=0.3))
    ax.text(0.5, 0.1, prediction, ha='center', fontsize=10, fontweight='bold',
            color='#E74C3C')
    ax.set_xlim(-0.2, 1.2)
    ax.set_ylim(-0.1, 1.3)

plt.tight_layout()
plt.savefig('../assets/autoregressive_generation.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Using Pretrained Models (BLIP)

In practice, you use pretrained models. BLIP is lightweight and works well.

In [ ]:
# Uncomment to run with pretrained BLIP (requires ~1GB download)
# from transformers import BlipProcessor, BlipForConditionalGeneration
# from PIL import Image
# import requests
#
# processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
# model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")
#
# url = "https://images.unsplash.com/photo-1574158622682-e40e69881006"
# image = Image.open(requests.get(url, stream=True).raw)
#
# inputs = processor(image, return_tensors="pt")
# output = model.generate(**inputs, max_length=50)
# caption = processor.decode(output[0], skip_special_tokens=True)
# print(f"Caption: {caption}")

print("To run pretrained BLIP, uncomment the cell above.")
print("It needs ~1GB download but works great on CPU!")
print("\nBlip models available:")
print("  - Salesforce/blip-image-captioning-base   (~990MB, good for CPU)")
print("  - Salesforce/blip-image-captioning-large  (~1.8GB, better quality)")

## Key Takeaways

1. **Image captioning = Vision Encoder + Text Decoder with Cross-Attention**
2. The decoder generates words **autoregressively** (one at a time)
3. **Cross-attention** lets each generated word look at relevant image regions
4. **Teacher forcing** during training: feed ground-truth tokens, predict next
5. For practical use: **BLIP** is lightweight and works on CPU

---
**Next:** `03_visual_question_answering.ipynb` - Answer questions about images